In [10]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util

df = pd.read_csv("Prajituri.csv", sep=",")

df["descriere"] = df.apply(
    lambda row: f"{row['denumire']}. Ingrediente principale: {row['ingrediente']}. Ingredient de baza: {row['Ingredient de baza']}. Textura: {row['Textura']}. Dulceata: {row['Dulceata']}. Provenienta: {row['Provenienta']}.",
    axis=1
)

model = SentenceTransformer("intfloat/multilingual-e5-large")

embeddings = model.encode(df["descriere"].tolist(), convert_to_tensor=True)

def recomanda_prajituri(preferinta, top_k=3):
    query_embedding = model.encode(preferinta, convert_to_tensor=True)
    cos_scores = util.cos_sim(query_embedding, embeddings)[0]
    top_results = cos_scores.topk(k=top_k)

    recomandari = []
    for score, idx in zip(top_results[0], top_results[1]):
        i = int(idx)
        recomandari.append((df.iloc[i]["denumire"], float(score)))
    return recomandari

print(recomanda_prajituri("vreau ceva răcoritor cu lămâie"))
print(recomanda_prajituri("ceva cu ciocolată și textură însiropată"))


C:\Users\tapuc\Desktop\handmade_backup\licenta\TransformereTest\.venv\Lib\site-packages\torch\nn\modules\module.py:1784: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


[('Tartaletta cu lămâie', 0.8370628952980042), ('Prajitura Andreea', 0.8212529420852661), ('Cornulete cu gem', 0.8207569122314453)]
[('Inghetata falsa', 0.851320743560791), ('Deliciu cu fructe', 0.8480017185211182), ('Prajitura cu ciocolata si visine', 0.8472384214401245)]


In [11]:
import pandas as pd

df = pd.read_csv("Prajituri.csv", sep=",")
print(df.columns.tolist())


['id', 'denumire', 'ingrediente', 'Ingredient de baza', 'Textura', 'Dulceata', 'Provenienta']


In [14]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np
import evaluate

# 1. Încarcă datele
dataset = load_dataset("csv", data_files={"train": "intents.csv"})["train"]

# Split train/test
dataset = dataset.train_test_split(test_size=0.2)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

# 2. Tokenizer
checkpoint = "dumitrescustefan/roberta-base-romanian-cased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
eval_dataset = eval_dataset.map(tokenize, batched=True)

# 3. Definire model
labels = sorted(list(set(train_dataset["categorie"])))
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

# 4. Antrenare
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
)

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()


OSError: dumitrescustefan/roberta-base-romanian-cased is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

KeyError: 'denumire'